In [0]:
%sql

CREATE OR REPLACE TABLE automobile_repair.gold.fact_orders_consolidated AS
WITH final_estimates AS (
  SELECT *
  FROM automobile_repair.silver.estimate
  WHERE version_type = 'Final'
),
revenue_agg AS (
  -- Aggregate multiple invoices per order
  SELECT 
    order_id,
    MAX(invoice_id) as invoice_id,
    MAX(invoice_date) as invoice_date,
    SUM(invoice_amount) as total_invoice_amount,
    MAX(manager_id) as manager_id,
    MAX(manager_name) as manager_name
  FROM automobile_repair.silver.invoice
  GROUP BY order_id
)
SELECT 
  -- Order Info
  o.order_id,
  o.store_id,
  o.service_type,
  o.order_status,
  
  -- Technician Info (embedded)
  o.technician_id,
  o.technician_name,
  
  -- Order Dates & Metrics
  o.vehicle_in_datetime,
  o.actual_work_start_datetime,
  o.actual_completion_datetime,
  o.vehicle_out_datetime,
  o.promised_delivery_datetime,
  o.actual_delivery_datetime,
  o.days_in_shop,
  o.delivery_accuracy_days,
  YEAR(o.vehicle_in_datetime) as order_year,
  MONTH(o.vehicle_in_datetime) as order_month,
  CAST(o.vehicle_in_datetime AS DATE) as order_date,
  
  -- Revenue Info
  r.invoice_id,
  r.invoice_date,
  r.total_invoice_amount,
  YEAR(r.invoice_date) as invoice_year,
  MONTH(r.invoice_date) as invoice_month,
  r.manager_id,
  r.manager_name,
  
  -- Budget Info
  b.budget_amount,
  
  -- Estimate Info (embedded estimator)
  e.estimate_id,
  e.estimator_id,
  e.estimator_name,
  e.estimate_type,
  e.initial_estimate,
  e.final_estimate,
  e.accuracy_score,
  e.variance_pct,
  e.estimate_date,
  
  -- Survey Info
  s.survey_id,
  s.survey_sent_date,
  s.survey_response_date,
  s.delivered_on_time_rating,
  s.work_quality_rating,
  s.cleanliness_rating,
  s.communication_rating,
  s.overall_satisfaction_rating,
  s.average_rating,
  s.responded_flag,
  
  current_timestamp() as created_at
  
FROM automobile_repair.silver.order o

-- Join Revenue (aggregated)
LEFT JOIN revenue_agg r ON o.order_id = r.order_id

-- Join Budget
LEFT JOIN automobile_repair.silver.budget b
  ON o.store_id = b.store_id
  AND YEAR(r.invoice_date) = b.budget_year
  AND MONTH(r.invoice_date) = b.budget_month

-- Join Final Estimates
LEFT JOIN final_estimates e ON o.order_id = e.order_id

-- Join Surveys
LEFT JOIN automobile_repair.silver.customer_survey s ON o.order_id = s.order_id

WHERE o.order_id IS NOT NULL